In [16]:
# =====================================================================
# Cell 1 — Install
# =====================================================================
!pip install -q torch torchvision timm wandb huggingface_hub tqdm numpy pyyaml pandas matplotlib


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [17]:
# =====================================================================
# Cell 2 — Config (FIXED)
# =====================================================================
import os, json, time, zipfile, shutil, random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from torchvision import datasets
from timm.data import Mixup, create_transform
from timm.loss import SoftTargetCrossEntropy
from timm.scheduler import CosineLRScheduler
from timm.models.layers import DropPath, trunc_normal_
from tqdm.auto import tqdm

DATA_ROOT    = "./data"
OUTPUT_DIR   = "./outputs/phase0_fixed"
EPOCHS       = 100
BATCH_SIZE   = 128
LR           = 1e-3          # ← was 2e-3, halved for stability
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS= 10            # ← was 5, longer warmup for graph models
GRAD_CLIP    = 1.0
NUM_WORKERS  = 8
NUM_CLASSES  = 100
SEED         = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Device: {DEVICE}")

Device: cuda


In [ ]:
# =====================================================================
# Cell 3 — HuggingFace login
# =====================================================================
HF_TOKEN = "hf_key"
login(token=HF_TOKEN)
print(f"✓ {whoami()['name']}")

✓ Ranjan637


In [ ]:
# =====================================================================
# Cell 4 — Download zips
# =====================================================================
def download_all_chunks(repo_id, raw_root, train_parts=5, val_parts=5):
    os.makedirs(raw_root, exist_ok=True)
    for split, n in [("train", train_parts), ("val", val_parts)]:
        for p in range(1, n + 1):
            fn = f"imagenet100_{split}_part{p}.zip"
            print(f"↓ {fn}")
            hf_hub_download(repo_id=repo_id, filename=fn,
                            repo_type="dataset", local_dir=raw_root)
    hf_hub_download(repo_id=repo_id, filename="Labels.json",
                    repo_type="dataset", local_dir=raw_root)
    print("Done.")

download_all_chunks(HF_REPO, RAW_ROOT)

In [ ]:
# =====================================================================
# Cell 5 — Extract + merge
# =====================================================================
def extract_and_merge(raw_root, data_root, train_parts=5, val_parts=5):
    tmp = os.path.join(raw_root, "_extract_tmp")
    os.makedirs(tmp, exist_ok=True)
    for split, n in [("train", train_parts), ("val", val_parts)]:
        split_dir = os.path.join(data_root, split)
        os.makedirs(split_dir, exist_ok=True)
        for p in range(1, n + 1):
            zp = os.path.join(raw_root, f"imagenet100_{split}_part{p}.zip")
            if not os.path.exists(zp): continue
            et = os.path.join(tmp, f"{split}_{p}")
            os.makedirs(et, exist_ok=True)
            with zipfile.ZipFile(zp) as zf: zf.extractall(et)
            for root, _, files in os.walk(et):
                if any(f.lower().endswith((".jpg",".jpeg",".png")) for f in files):
                    cls = os.path.basename(root)
                    dest = os.path.join(split_dir, cls)
                    os.makedirs(dest, exist_ok=True)
                    for f in files:
                        if f.lower().endswith((".jpg",".jpeg",".png")):
                            d = os.path.join(dest, f)
                            if not os.path.exists(d):
                                shutil.move(os.path.join(root, f), d)
    shutil.rmtree(tmp, ignore_errors=True)
    print("Done.")

extract_and_merge(RAW_ROOT, DATA_ROOT)

In [19]:
# =====================================================================
# Cell 6 — Verify
# =====================================================================
for split in ["train","val"]:
    d = os.path.join(DATA_ROOT, split)
    cls = [c for c in os.listdir(d) if os.path.isdir(os.path.join(d,c))]
    n = sum(len(f) for _,_,f in os.walk(d))
    print(f"{split}: {len(cls)} classes, {n} images")

train: 100 classes, 128982 images
val: 100 classes, 5000 images


In [22]:
# =====================================================================
# Cell 7 — FIXED MODEL (correct aggregation, float32, max-relative)
# =====================================================================

class LayerNorm2d(nn.Module):
    def __init__(self, ch, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ch))
        self.bias   = nn.Parameter(torch.zeros(ch))
        self.eps = eps
    def forward(self, x):
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        y = (x - u) / torch.sqrt(s + self.eps)
        return self.weight[:, None, None] * y + self.bias[:, None, None]


class MRConv2d(nn.Module):
    """
    Max-Relative Graph Convolution — FIXED:
      1. Correct aggregation: max over neighbors PER CENTER NODE
      2. Correct direction: x_neighbor - x_center (not reversed)
      3. Entire forward in float32 (AMP-safe via decorator)
      4. Concatenate [center, max_relative] before projection (matches paper)
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        # Concat center (in_ch) + max_relative (in_ch) → project to out_ch
        self.fc = nn.Linear(in_ch * 2, out_ch)

    @torch.amp.autocast('cuda', enabled=False)     # ← FIX 3: force float32
    def forward(self, x, edge_index, k):
        """
        x:          (B*N, C) node features
        edge_index: (2, B*N*k) where [0]=center, [1]=neighbor
        k:          number of neighbors per node
        """
        x = x.float()                               # ← FIX 3: ensure float32
        row, col = edge_index                        # row=center, col=neighbor

        # Relative features: how each neighbor differs from center
        x_rel = x[col] - x[row]                     # ← FIX 1: was x[row]-x[col]

        # Max aggregation per center node
        # Edges are ordered: k consecutive entries per center (guaranteed by kNN)
        BN = x.size(0)
        x_rel = x_rel.view(BN, k, -1)               # (B*N, k, C)
        x_max = x_rel.max(dim=1)[0]                  # ← FIX 2: max, not sum

        # Concatenate center features + max-relative, then project
        out = self.fc(torch.cat([x, x_max], dim=-1)) # (B*N, out_ch)
        return out


class Grapher(nn.Module):
    """Pre-norm Grapher block with kNN graph construction."""
    def __init__(self, ch, k=9, drop_path=0.0):
        super().__init__()
        self.k, self.ch = k, ch

        # Pre-norm sublayer 1: graph conv
        self.norm1 = LayerNorm2d(ch)
        self.gcn   = MRConv2d(ch, ch)

        # Pre-norm sublayer 2: FFN
        self.norm2 = LayerNorm2d(ch)
        self.ffn = nn.Sequential(
            nn.Conv2d(ch, ch * 4, 1, bias=False),
            nn.GELU(),
            nn.Conv2d(ch * 4, ch, 1, bias=False),
        )
        self.drop = DropPath(drop_path) if drop_path > 0 else nn.Identity()

    def _knn(self, x):
        B, C, H, W = x.shape
        N = H * W
        ke = min(self.k, N - 1)
        with torch.amp.autocast('cuda', enabled=False):
            xf = x.permute(0,2,3,1).reshape(B, N, C).float()
            xn = F.normalize(xf, dim=-1, eps=1e-6)
            sim = torch.bmm(xn, xn.transpose(1, 2))
            sim.diagonal(dim1=1, dim2=2).fill_(-float('inf'))
            _, idx = sim.topk(ke, dim=-1)

        base = torch.arange(B, device=x.device).view(B,1,1) * N
        # row = center nodes, col = neighbor nodes
        row = (torch.arange(N, device=x.device).view(1,N,1) + base).expand(B,N,ke)
        col = idx + base.expand(B,N,ke)
        ei = torch.stack([row.reshape(-1), col.reshape(-1)], 0)
        return ei, B, N, ke       # ← also return actual k used

    def forward(self, x):
        B, C, H, W = x.shape
        # --- sublayer 1: graph conv ---
        h = self.norm1(x)
        ei, Bg, N, ke = self._knn(h)
        h_flat = h.permute(0,2,3,1).reshape(Bg * N, C)
        h_conv = self.gcn(h_flat, ei, ke)            # ← pass k
        h_conv = h_conv.reshape(Bg, H, W, C).permute(0,3,1,2)
        x = x + self.drop(h_conv)
        # --- sublayer 2: FFN ---
        h = self.norm2(x)
        x = x + self.drop(self.ffn(h))
        return x


class PyramidViG(nn.Module):
    def __init__(self, num_classes=1000, embed_dim=48,
                 depths=(2,2,6,2), drop_path_rate=0.1):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, embed_dim, 16, 16, bias=False),
            LayerNorm2d(embed_dim),
            nn.GELU(),
        )
        self.stages = nn.ModuleList()
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        cur = 0
        for i, d in enumerate(depths):
            dim = embed_dim * (2**i)
            if i > 0:
                ds = nn.Sequential(
                    nn.Conv2d(dim//2, dim, 2, 2, bias=False),
                    LayerNorm2d(dim),
                )
            else:
                ds = nn.Identity()
            stage = nn.Sequential(
                ds,
                *[Grapher(dim, k=9, drop_path=dpr[cur+j]) for j in range(d)]
            )
            self.stages.append(stage)
            cur += d
        self.norm = nn.LayerNorm(embed_dim * 8)
        self.head = nn.Linear(embed_dim * 8, num_classes)
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None: nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.LayerNorm, LayerNorm2d)):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x):
        x = self.stem(x)
        for s in self.stages:
            x = s(x)
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.head(self.norm(x))


def pyramid_vig_ti(num_classes=100):
    return PyramidViG(num_classes=num_classes, embed_dim=48,
                      depths=(2,2,6,2), drop_path_rate=0.1)

_m = pyramid_vig_ti(NUM_CLASSES)
print(f"Params: {sum(p.numel() for p in _m.parameters() if p.requires_grad)/1e6:.2f}M")
del _m; torch.cuda.empty_cache()

Params: 5.87M


In [7]:
# =====================================================================
# Cell 8 — Data loaders
# =====================================================================
def build_loaders(root, bs=128, nw=8):
    tr_t = create_transform(224, True, color_jitter=0.4,
        auto_augment='rand-m9-mstd0.5-inc1', interpolation='bicubic',
        re_prob=0.25, re_mode='pixel', re_count=1)
    vl_t = create_transform(224, False, interpolation='bicubic', crop_pct=0.95)
    tr = datasets.ImageFolder(os.path.join(root,"train"), tr_t)
    vl = datasets.ImageFolder(os.path.join(root,"val"),   vl_t)
    tr_l = DataLoader(tr, bs, True, num_workers=nw, pin_memory=True,
                      drop_last=True, persistent_workers=nw>0)
    vl_l = DataLoader(vl, bs, False, num_workers=nw, pin_memory=True,
                      persistent_workers=nw>0)
    return tr_l, vl_l, len(tr.classes)

train_loader, val_loader, n_classes = build_loaders(DATA_ROOT, BATCH_SIZE, NUM_WORKERS)
print(f"Train: {len(train_loader)} batches | Val: {len(val_loader)} batches")

Train: 1007 batches | Val: 40 batches


In [8]:
# =====================================================================
# Cell 9 — DIAGNOSTIC: overfit test on 5 batches
# =====================================================================
# If the model cannot overfit 5 batches → architecture is broken.
# This takes ~1 minute on an H200.

print("="*70); print("DIAGNOSTIC: Overfit 5 batches"); print("="*70)

diag = pyramid_vig_ti(NUM_CLASSES).to(DEVICE)
opt  = optim.AdamW(diag.parameters(), lr=2e-3, weight_decay=0.0)
lf   = nn.CrossEntropyLoss()
sc   = GradScaler('cuda')

batches = [b for i, b in enumerate(train_loader) if i < 5]
print(f"5 batches = {sum(b[0].size(0) for b in batches)} samples, "
      f"{NUM_CLASSES} classes")

for ep in range(30):
    diag.train()
    for imgs, tgts in batches:
        imgs, tgts = imgs.to(DEVICE), tgts.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with autocast('cuda'):
            loss = lf(diag(imgs), tgts)
        sc.scale(loss).backward()
        sc.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(diag.parameters(), 1.0)
        sc.step(opt); sc.update()
    if (ep+1) % 5 == 0:
        diag.eval()
        c = t = 0
        with torch.no_grad(), autocast('cuda'):
            for imgs, tgts in batches:
                imgs, tgts = imgs.to(DEVICE), tgts.to(DEVICE)
                c += (diag(imgs).argmax(1) == tgts).sum().item()
                t += tgts.size(0)
        print(f"Epoch {ep+1:2d}: overfit train acc = {c/t:.4f}")

del diag, opt, sc; torch.cuda.empty_cache()

DIAGNOSTIC: Overfit 5 batches
5 batches = 640 samples, 100 classes
Epoch  5: overfit train acc = 0.6391
Epoch 10: overfit train acc = 0.9625
Epoch 15: overfit train acc = 0.9609
Epoch 20: overfit train acc = 0.9531
Epoch 25: overfit train acc = 0.9531
Epoch 30: overfit train acc = 0.9547


In [23]:
# =====================================================================
# Cell 10 — FIXED training loop with NaN detection + grad monitoring
# =====================================================================

def train_epoch(model, loader, opt, sc, crit, mixup, device):
    model.train()
    total_loss = 0.0
    total_samples = 0
    grad_norms = []

    for imgs, tgts in tqdm(loader, desc="Train", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        tgts = tgts.to(device, non_blocking=True)
        if mixup is not None:
            imgs, tgts = mixup(imgs, tgts)

        opt.zero_grad(set_to_none=True)
        with autocast('cuda'):
            loss = crit(model(imgs), tgts)

        # ---- NaN DETECTION ----
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  ⚠️  NaN/Inf loss detected! Skipping batch.")
            opt.zero_grad(set_to_none=True)
            continue

        sc.scale(loss).backward()
        sc.unscale_(opt)

        # ---- GRADIENT NORM MONITORING ----
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        grad_norms.append(gn.item() if isinstance(gn, torch.Tensor) else gn)

        sc.step(opt)
        sc.update()
        total_loss += loss.item() * imgs.size(0)
        total_samples += imgs.size(0)

    avg_loss = total_loss / max(total_samples, 1)
    avg_gn   = np.mean(grad_norms) if grad_norms else 0.0
    max_gn   = max(grad_norms) if grad_norms else 0.0
    return avg_loss, avg_gn, max_gn


@torch.no_grad()
def validate(model, loader, crit, device):
    model.eval()
    total_loss = correct = total = 0
    for imgs, tgts in tqdm(loader, desc="Val", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        tgts = tgts.to(device, non_blocking=True)
        with autocast('cuda'):
            out = model(imgs)
            total_loss += crit(out, tgts).item() * imgs.size(0)
        correct += (out.argmax(1) == tgts).sum().item()
        total   += tgts.size(0)
    return total_loss / total, correct / total


def run_experiment_0_1_fixed():
    print("=" * 80)
    print("EXPERIMENT 0.1 (FIXED): ViG-Ti on ImageNet-100")
    print("=" * 80)

    model = pyramid_vig_ti(num_classes=NUM_CLASSES).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Params: {n_params / 1e6:.2f}M")

    # -------- Mixup with REDUCED augmentation --------
    mixup = Mixup(
        mixup_alpha=0.2,            # ← was 0.8 (too strong for 5M model)
        cutmix_alpha=0.2,           # ← was 1.0
        cutmix_minmax=None,
        prob=0.5,                   # ← was 1.0 (apply only 50% of the time)
        switch_prob=0.5,
        mode='batch',
        label_smoothing=0.1,
        num_classes=NUM_CLASSES,
    )

    tr_crit = SoftTargetCrossEntropy()
    vl_crit = nn.CrossEntropyLoss()

    # -------- Optimizer with param groups --------
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if p.ndim <= 1 or 'norm' in name.lower() or 'bias' in name.lower():
            no_decay.append(p)
        else:
            decay.append(p)
    opt = optim.AdamW(
        [{'params': decay,    'weight_decay': WEIGHT_DECAY},
         {'params': no_decay, 'weight_decay': 0.0}],
        lr=LR,
    )

    sc  = GradScaler('cuda')
    sch = CosineLRScheduler(
        opt, t_initial=EPOCHS, lr_min=1e-6,
        warmup_t=WARMUP_EPOCHS, warmup_lr_init=1e-6,
    )
    sch.step(0)

    best = 0.0
    hist = []
    nan_count = 0

    for ep in range(EPOCHS):
        t0 = time.time()
        tl, avg_gn, max_gn = train_epoch(
            model, train_loader, opt, sc, tr_crit, mixup, DEVICE
        )
        vl, va = validate(model, val_loader, vl_crit, DEVICE)
        sch.step(ep + 1)
        dt = time.time() - t0

        hist.append(dict(
            epoch=ep+1, train_loss=tl, val_loss=vl, val_acc=va,
            lr=opt.param_groups[0]['lr'], time=dt,
            avg_grad_norm=avg_gn, max_grad_norm=max_gn,
        ))

        flag = ""
        if va > best:
            best = va
            torch.save(model.state_dict(),
                       os.path.join(OUTPUT_DIR, "best_vig_ti.pth"))
            flag = f" ★ new best"

        print(f"Epoch {ep+1:3d}/{EPOCHS} | "
              f"TrL {tl:.4f} | VaL {vl:.4f} | Acc {va:.4f} | "
              f"GN {avg_gn:.2f}/{max_gn:.2f} | "
              f"LR {opt.param_groups[0]['lr']:.2e} | "
              f"{dt:.1f}s{flag}")

        # ---- EARLY ABORT on persistent NaN ----
        if math.isnan(tl):
            nan_count += 1
            if nan_count >= 3:
                print("❌ 3 consecutive NaN epochs. Aborting training.")
                break
        else:
            nan_count = 0

    print(f"\nBest val acc: {best:.4f}")
    pd.DataFrame(hist).to_csv(
        os.path.join(OUTPUT_DIR, "history.csv"), index=False
    )
    return best, hist


# =====================================================================
# Cell 11 — Run
# =====================================================================
best_acc, history = run_experiment_0_1_fixed()

EXPERIMENT 0.1 (FIXED): ViG-Ti on ImageNet-100
Params: 5.87M


RuntimeError: cannot reshape tensor of 0 elements into shape [128, 0, -1] because the unspecified dimension size -1 can be any value and is ambiguous

In [14]:
# =====================================================================
# Cell 11 — Run
# =====================================================================
best_acc, history = run_experiment_0_1()

EXPERIMENT 0.1: ViG-Ti on ImageNet-100
Params: 5.33M
[check] model output: (128, 100)  (expected: (B, 100))
[check] mixup targets: (128, 100)  (expected: (B, 100))


Epoch   1/100 | Tr 4.6775 | Va 4.6724 Acc 0.0106 | 72.7s
  → new best 0.0106


Epoch   2/100 | Tr 4.6402 | Va 4.5711 Acc 0.0196 | 73.8s
  → new best 0.0196


Epoch   3/100 | Tr 4.6145 | Va 4.6025 Acc 0.0130 | 75.9s


Epoch   4/100 | Tr 4.6089 | Va 4.6099 Acc 0.0086 | 73.5s


Epoch   5/100 | Tr 4.6079 | Va 4.6083 Acc 0.0090 | 73.7s


... (91 outputs left out)

Epoch  97/100 | Tr nan | Va nan Acc 0.0100 | 71.4s


Epoch  98/100 | Tr nan | Va nan Acc 0.0100 | 74.6s


Epoch  99/100 | Tr nan | Va nan Acc 0.0100 | 75.3s


Epoch 100/100 | Tr nan | Va nan Acc 0.0100 | 76.4s

Best val acc: 0.0196
